In [ ]:
!pip install datasets
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.6/474.6 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 19.9 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 59.4 MB/s eta 0:00

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback, TextClassificationPipeline

In [ ]:
tokenizer = BertTokenizer.from_pretrained('jhgan/ko-sroberta-multitask')
model = BertForSequenceClassification.from_pretrained("jhgan/ko-sroberta-multitask")
# text = "Replace me by any text you'd like."
# encoded_input = tokenizer(text, return_tensors='pt')
# output = model(**encoded_input)

You are using a model of type roberta to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at jhgan/ko-sroberta-multitask and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd
import torch
from datasets import load_metric

In [ ]:
open('안유진.txt', 'r',encoding='utf-8').read()[:100]

'document\tlabel\n"화성시, 도민체전 사전경기서 1부 선두 질주"\t0\n전날 여자부 접영 100m 결승에서 1분00초52로 금메달을 획득한 이근희는 이날 여자부 접영 50m'

In [ ]:
import pandas as pd
import numpy as np

# CSV 파일 읽기
data = pd.read_csv('안유진.txt',sep='\t')
data = data.dropna()

# 데이터 인덱스 리스트 생성
indices = np.arange(len(data))

# train set과 test set 분할
train_indices = np.random.choice(indices, size=int(0.8 * len(data)), replace=False)
test_indices = np.setdiff1d(indices, train_indices)

# train set과 test set 생성
train_set = data.iloc[train_indices]
test_set = data.iloc[test_indices]




In [ ]:
# train set 확인
print(train_set.shape)
print(train_set.head())



(4570, 2)
                                               document  label
1059  이은지는 “언니 잘 됐다”를 안유진, 이영지, 미미의 성대모사로 전해 주위를 폭소케...    1.0
5653  거다 안유진.....유진씨...유진아아아....... 어쩌자고 그리 어여쁘냐.......    1.0
1046      [SC현장] 아이브X르세라핌 기다릴 것…가요광장 DJ 이은지, 통통 튀는 인...    0.0
244                   경기체고 박선우·이서진, 동아수영대회에서 나란히 금메달 획득    0.0
5660                       아이즈원 - 라비앙로즈 뮤비 움짤/ 장원영, 안유진    1.0


In [ ]:
# test set 확인
print(test_set.shape)
print(test_set.head())


(1143, 2)
                                             document  label
8                                ‘K-909’ 아이브, 치명적인 변신    1.0
11  안유진은 설계자 콘셉트로 색다른 스타일을 선보였고, 먼저 ‘일레븐과 러브 다이브’ ...    1.0
17  아이브(안유진·가을·레이·장원영·리즈·이서)가 오직 K-909만을 위한 무대를 선물...    1.0
18                   K-909 아이브, 치명적 변신…골든디스크 무대 최초 공개    1.0
24                 아이브, 오늘(6일) K-909 출격‥골든디스크 무대 첫 공개    1.0


- Train-test split

In [ ]:
# from skmultilearn.model_selection import iterative_train_test_split ### For multi-label split
# X_val, val_labels, X_test, test_labels = iterative_train_test_split(X_test, test_labels, test_size = 0.5)
# from sklearn.model_selection import train_test_split

- Tokeninzing

In [ ]:
train_ = train_set['document'].to_list()[:15000]
# train_ = ratings_train['document'].to_list()
train_label = train_set['label'].to_numpy()[:15000]
# train_label = ratings_train['label'].to_numpy()

In [ ]:
train_data  = tokenizer(train_, return_tensors='pt', truncation = True, padding = True)

In [ ]:
test_ = test_set['document'].to_list()[:5000]
test_data  = tokenizer(test_, return_tensors='pt', truncation=True, padding=True)
test_label = test_set['label'].to_numpy()[:5000]

In [ ]:
from torch.utils.data import Dataset
class PrepDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encoding = encodings
        self.labels = labels

    def __getitem__(self, idx):
        data = {key: val[idx] for key, val in self.encoding.items()}
        data['labels'] = torch.tensor(self.labels[idx]).long()
        return data

    def __len__(self):
        return len(self.labels)

In [ ]:
train = PrepDataset(train_data, train_label)
test  = PrepDataset(test_data, test_label)

In [ ]:
training_args = TrainingArguments(
    output_dir='./checkpoint',          # output directory
    num_train_epochs = 5,              # total number of training epochs
    per_device_train_batch_size = 16,  # batch size per device during training
    per_device_eval_batch_size  = 16,   # batch size for evaluation
    weight_decay = 0.0001,               # strength of weight decay
    logging_dir='./log',     
    do_eval=True,
    save_strategy='steps',
    evaluation_strategy = 'steps',
    fp16 = True,
    learning_rate = 2e-5,
    log_level='info',
    eval_steps = 100,
    load_best_model_at_end = True
)

In [ ]:
# def compute_metrics(self, pred):
#       labels = pred.label_ids
#       preds = pred.predictions.argmax(-1)
#       m1 = load_metric('accuracy')
#       # m2 = load_metric('f1')
#       acc = m1.compute(predictions = preds, references = labels)['accuracy']
#       # f1  = m2.compute(predictions = preds, references = labels)['f1']
#       return {'accuracy':acc}

In [ ]:
trainer = Trainer(
    model = model,
    args  = training_args,
    train_dataset   = train, # 학습 세트
    eval_dataset    = test, # 테스트 세트
    # compute_metrics = compute_metrics, # metric 계산 함수
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 5)]
)

Using cuda_amp half precision backend


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 4,570
  Num Epochs = 5
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 1,430
  Number of trainable parameters = 110,619,650


Step,Training Loss,Validation Loss
100,No log,0.325099
200,No log,0.317537
300,No log,0.291602
400,No log,0.376174
500,0.293500,0.362435
600,0.293500,0.342388
700,0.293500,0.416815
800,0.293500,0.385915
900,0.293500,0.465231
1000,0.153700,0.481650


***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
Saving model checkpoint to ./checkpoint/checkpoint-500
Configuration saved in ./checkpoint/checkpoint-500/config.json
Model weights saved in ./checkpoint/checkpoint-500/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
***** Running Evaluation *****
  Num examples = 1143
  Batch size = 16
Saving model checkpoint to ./checkpoint/checkpoint-1000
Configuration saved in ./checkpoint/checkpoint-

TrainOutput(global_step=1100, training_loss=0.21573984319513495, metrics={'train_runtime': 227.3155, 'train_samples_per_second': 100.521, 'train_steps_per_second': 6.291, 'total_flos': 1427560419733680.0, 'train_loss': 0.21573984319513495, 'epoch': 3.85})

여기부터는 기존에 만든 모델에 적용해보는 부분

In [ ]:
text_classifier = TextClassificationPipeline(
    tokenizer = tokenizer, 
    model     = model,
    device    = 'cuda:0'
)

Xformers is not installed correctly. If you want to use memorry_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [ ]:
text_classifier = TextClassificationPipeline(
    tokenizer = tokenizer, 
    model     = model, 
    device    = 'cuda:0',
    # top_k     = None
)


In [ ]:
# 새로운 데이터 생성

import os
import sys
import urllib.request
import json

ID = "E4_pVsh0x3azbPhb0zTR" 
KEY = "STUh3EX0rD"

def naver_api(where, what, how) : # where : 뉴스,article, blog, what : 우리가 찾을 검색어, how : 어떤 파일로 써줄지
    
    encText = urllib.parse.quote(what)

    for i in range(1,1001,100):
        url = "https://openapi.naver.com/v1/search/" + where + "?query=" + encText + '&display=100' + "&start=" + str(i)
 
        request = urllib.request.Request(url)
        request.add_header("X-Naver-Client-Id",ID)
        request.add_header("X-Naver-Client-Secret",KEY)
        response = urllib.request.urlopen(request)
        rescode = response.getcode()
        
        if(rescode==200):
            response_body = response.read()
            results = response_body.decode('utf-8')
            items = json.loads(results)['items']

            for item in items :
                f = open(how,'a',encoding = 'utf8')
                title = item['title']
                description = item['description']
                
                title = title.replace('<b>','')
                title = title.replace('</b>','')
                title = title.replace('&apos;','')
                title = title.replace('&quot;','')
                description = description.replace('<b>','')
                description = description.replace('</b>','')
                description = description.replace('&apos;','')
                description = description.replace('&quot;','')

                f.write(title + '\n')
                f.write(description +'\n')
                f.close()


In [ ]:
# 새로운 데이터 생성(장원영)
naver_api('news','장원영','장원영.txt')

In [ ]:
data1 = pd.read_csv('장원영.txt',sep='\t')

In [ ]:
# data안의 문장들을 리스트 입력
data_list = data1.iloc[:5000].values.tolist()
data_list

In [ ]:
# labe==1인 값을 리스트 안에 모음
label_1_sentences = []
for i in range(1, len(data_list)):
    result = text_classifier(data_list[i])
    if result[0]['label'] == 'LABEL_1':
        label_1_sentences.append(data_list[i])

# 결과 확인(LABEL==1인 문장들이 담겨 있는 리스트)
label_1_sentences



In [ ]:
# kkma 설치
# !pip install konlpy
# from konlpy.tag import 
# kkma = Kkma()

# def morph(input_data) : #형태소 분석
#     preprocessed = kkma.pos(input_data)
#     print(preprocessed)

In [ ]:
from konlpy.tag import Hannanum
from collections import Counter

hannanum = Hannanum()

words = []

for sentence in label_1_sentences:
    tagged_words = hannanum.pos(str(sentence))
    for word, tag in tagged_words:
        if tag.startswith('N'):
            words.append(word)

word_counts = Counter(words)

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

for word, count in sorted_words:
    print(word, count)



In [ ]:
# 2번째 데이터
naver_api('news','카리나','카리나.txt')
data1 = pd.read_csv('카리나.txt',sep='\t')
data_list = data1.iloc[:10000].values.tolist()
data_list
label_1_sentences = []
for i in range(1, len(data_list)):
    result = text_classifier(data_list[i])
    if result[0]['label'] == 'LABEL_1':
        label_1_sentences.append(data_list[i])

words = []

for sentence in label_1_sentences:
    tagged_words = hannanum.pos(str(sentence))
    for word, tag in tagged_words:
        if tag.startswith('N'):
            words.append(word)

word_counts = Counter(words)

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

for word, count in sorted_words:
    print(word, count)        

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1080: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
‘본스타미디어제작센터’ 2
정진솔 2
위키미키 2
정해림 2
“쇼파드 2
그랑프리 2
최초… 2
나이비스 2
체감하시 2
닭볶음탕 2
퐁당퐁담 2
음방을 2
오픈…JCL 2
허영지 2
됐다(정희 2
['영앙제 2
음방하다 2
음방하 2
리스펙트 2
“‘Spicy’ 2
속설 2
닫아”(정희 2
찐팬 2
극장 2
펜트하우스 2
킹받 2
“나이비스 2
수록”(정희 2
존경해 2
않았냐 2
목금토일 2
MBCFM4U 2
음방, 2
돼”(정희 2
신인 2
너스레 2
영앙제 2
에스파♥ 2
칼단발 2
=선미, 2
정호연, 2
고민시 2
무더위 2
=선미 2
복숭아 2
피치 2
로즈 2
‘Spicy’, 2
‘관심↑’ 2
맛 2
['바비인형 2
싱크로율 2
‘인간 2
바비인형’ 2
신축성 2
타이트 2
자카드 2
문양 2
영화제行 2
선다‥K팝 2
간다…K팝 2
['“극강 2
스캣”…말로·박라온·강윤미·김민희 2
['말로·박라온·강윤미·김민희 2
16일(현지시각) 2
휴양 2
도시 2
제75회 2
[사진=쇼파드] 2
53,570 2
53,570장 2
['앰버서더 2
고혹美 2
주최·주관사 2
마포구 2
매치’(GOOD 2
양성 2
본스타, 2
키즈전문제작사 2
잇츠 2
예고…라이브 2
['효연 2
김종민 2
하고파…母 2
최애=장원영 2
['다시, 2
셋째 2
엄정화 2
댄스가수 2
유랑단 2
전국 2
유리X효연 2
소시 2
멋있어 2
2순위 2
유리 2
어머님 2
태교 2
가요 2
['아이브→르세라핌→에스파→(여자)아이들, 2
전성시대 2
일주 2
블록체인 2
생태계…내 2
[김주완 2
프로젝트 2
톱(Girls 2
수(128만 2
모드하우스 2
['초인플레 2
인기몰이 2
‘아르헨티나판 2
트럼프’[원호연 2
1위(389,369장)…NCT 2
8,946장 2
1,591장 2
에스쿱스·정한·조슈아·준·호시·원우·우지·디에잇 2
형태 2
ⓒ투데이신문 2
80to23 2
김범성(광주화정초·6학년)군 2
15주년차

In [ ]:
# 3번째 데이터
naver_api('news','김채원','김채원.txt')
data1 = pd.read_csv('김채원.txt',sep='\t')
data_list = data1.iloc[:10000].values.tolist()
data_list
label_1_sentences = []
for i in range(1, len(data_list)):
    result = text_classifier(data_list[i])
    if result[0]['label'] == 'LABEL_1':
        label_1_sentences.append(data_list[i])

words = []

for sentence in label_1_sentences:
    tagged_words = hannanum.pos(str(sentence))
    for word, tag in tagged_words:
        if tag.startswith('N'):
            words.append(word)

word_counts = Counter(words)

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

for word, count in sorted_words:
    print(word, count)        

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1080: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


르세라핌 500
김채원 489
사쿠라 437
김채원, 403
카즈하, 398
허윤진, 385
홍은채) 330
['르세라핌, 250
정규 203
1집 194
['르세라핌 190
1위 175
그룹 169
르세라핌(김채원, 167
서울 167
오전 161
주간 140
파트리샤 136
5월 132
방송 129
참석 129
톱 123
미연 118
앨범 117
뮤직뱅크 114
리정, 112
진행 110
최예나, 109
르세라핌(LE 105
홍은채 105
걸그룹 102
혜 100
오리콘 98
이 97
여자 97
신곡 95
등 94
아이들 91
차트 90
공개 86
일본 84
발매 82
허윤진 81
멤버 80
음악 80
이날 77
여의 77
['르세라핌(김채원, 76
판매량 76
블랙핑크 75
포토 75
혜미리예채파 73
리허설 69
영상 68
글로벌 67
예능 67
빌보드 64
활동 63
멤버들 62
랭킹 61
19일 59
영등포구 58
‘UNFORGIVEN’ 57
무대 57
진입 55
음반 54
스포티파 52
발표 52
유튜브 51
출연 51
가수 51
연속 51
스포티파이 51
2023. 51
별관 51
카즈하 50
마지막 50
차트(5월 50
한터차트 50
12일 50
첫 49
최대 49
최신 49
팝 49
세계 48
수 48
‘UNFORGIVEN 47
포즈 47
정상 47
하이브 46
스트리밍 46
05. 46
후속곡 45
지수 45
미국 45
퍼포먼스 45
2023 45
여의도동 45
채널 44
업체 44
‘주간 43
Rodgers)’ 42
['혜미리예채파 42
오후 41
이번 41
출근길 41
1일 40
르세라핌, 40
집계 40
28일 39
판매 39
15일 39
신관 39
공개홀 39
프시케 38
뮤직비디오 38
소속사 38
국 38
13일 38
장 37
지역 37
사이트 37
6위 36
만 36
일정 36
日 36
데뷔 35
시작 35
게재 35
['르세라핌(LE 35
퀘스트 35
김포국제공항 35
12 34
장군엔터테인먼트 34
수염 33
미모 33
타이

In [ ]:
pd.DataFrame(text_classifier(["오늘 영화 너무 재미있었어요.", "오늘 영화 정말 별로에요."]))

,label,score
0,LABEL_1,0.749115
1,LABEL_0,0.818968


In [ ]:
result = text_classifier(test_)

In [ ]:
rlt = [int(i['label'][-1]) for i in result]

In [ ]:
### result --> 숫자 뽑아서 리스트로

In [ ]:
rt = ratings_test.head(5000).copy()
print(rt.dtypes)
rt.head()

In [ ]:
rt['fit'] = rlt

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true=rt['label'], y_pred=rt['fit']))

In [ ]:
### 책에서는 85.52%

In [ ]:
!pip install diffusers

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

prompt = "a photo of an astronaut riding a horse on mars"
image = pipe(prompt).images[0]  
    
image.save("astronaut_rides_horse.png")